# STAC Catalog Builder — Crop productivity correction (GeoParquet)

Publish salinity-corrected crop productivity GeoParquet to the Global Coastal Atlas STAC.

| Step | Description |
|------|-------------|
| 1. Configure paths | Local parquet + metadata, GCS targets |
| 2. Helper functions | Collection / item builders (parquet assets) |
| 3. Create collection | From metadata JSON |
| 4. Table asset metadata | `table:columns` descriptions |
| 5. Test one item | Validate before batch |
| 6. Build all items | One item per parquet store |
| 7. Save STAC locally | `STAC/data/current/` |
| 8. Upload parquet to GCS | Optional |
| 9. Upload STAC to GCS | Optional |

**Prerequisites:** run `notebooks/13_crop_productivity_correction.ipynb` first (CSV → GeoParquet + metadata).

Structure follows `12_crop_production.ipynb`; parquet item pattern inspired by [coclicodata NUTS0 STAC](https://github.com/openearth/coclicodata/blob/main/scripts/create_stacs/99_NUTS0_CM_stacs.py).


In [1]:
import datetime
import json
import os
import sys
from pathlib import Path
from posixpath import join as urljoin

import geopandas as gpd
import pandas as pd
import pystac
import shapely.geometry
from IPython.display import display
from pystac.stac_io import DefaultStacIO

from coclicodata.coclico_stac.extension import (
    CoclicoExtension,
    CollectionCoclicoExtension,
)
from coclicodata.coclico_stac.templates import get_template_collection
from coclicodata.etl.cloud_utils import (
    dir_to_google_cloud,
    file_to_google_cloud,
    load_google_credentials,
)

# Optional: table extension helpers from coastmonitor (same repo family as GCA)
REPO_ROOT = Path(r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories")
sys.path.insert(0, str(REPO_ROOT / "coastmonitor" / "src"))
try:
    from coastmonitor import stac_table  # noqa: F401

    HAS_STAC_TABLE = True
except Exception:
    HAS_STAC_TABLE = False

PARQUET_MEDIA_TYPE = "application/vnd.apache.parquet"
print("stac_table available:", HAS_STAC_TABLE)


stac_table available: True


## 1) Configure paths and options

Point `data_dir` at the GeoParquet folder produced by the notebooks/13 preprocess step.


In [2]:
repo_root = Path(r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories")
stac_dir = repo_root / "global-coastal-atlas/STAC/data/current"

data_dir = Path(
    r"P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_productivity_correction"
)
metadata_path = data_dir / "metadata_crop_productivity_correction.json"
google_cred_path = Path(
    r"P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\google_credentials.json"
)

PARQUET_STORES = [
    {
        "id": "baseline",
        "filename": "parquets/baseline/corrected_yield.parquet",
        "title": "Corrected crop yield (baseline)",
        "description": (
            "Salinity-corrected crop production by Mekong freshwater zone / year"
        ),
    },
]

gcs_protocol = "https://storage.googleapis.com"
gcs_project = "GCA - 11210264"
bucket_name = "gca-data-public"
bucket_proj = "gca"
stac_cloud_name = "gca-stac-7"

# Coclico frontend props (map / table product — no station line plot)
PLOT_TYPE = "map"
PLOT_X_AXIS = ""
PLOT_SERIES = ""

if not metadata_path.is_file():
    raise FileNotFoundError(f"Metadata not found: {metadata_path}")

with open(metadata_path, encoding="utf-8") as f:
    metadata = json.load(f)

REQUIRED_METADATA_KEYS = [
    "SPATIAL_EXTENT",
    "TEMPORAL_EXTENT",
    "COLLECTION_ID",
    "TITLE",
    "DESCRIPTION",
    "LICENSE",
    "PROVIDERS",
    "KEYWORDS",
    "UNITS",
    "MEDIA_TYPE",
]
missing_keys = [key for key in REQUIRED_METADATA_KEYS if key not in metadata]
if missing_keys:
    raise KeyError(
        f"Missing required metadata keys in {metadata_path}: {", ".join(missing_keys)}"
    )

collection_id = metadata["COLLECTION_ID"]
proj_name = collection_id
href_prefix = urljoin(gcs_protocol, bucket_name, bucket_proj, proj_name)

for store in PARQUET_STORES:
    parquet_path = data_dir / store["filename"]
    if not parquet_path.exists():
        raise FileNotFoundError(f"Parquet not found: {parquet_path}")

print("Parquet input:", data_dir)
print("STAC output:", stac_dir / collection_id)
print("Collection id:", collection_id)
print("Parquet stores:", [s["filename"] for s in PARQUET_STORES])
print("Temporal extent:", metadata["TEMPORAL_EXTENT"])
print("Spatial extent:", metadata["SPATIAL_EXTENT"])
print("MEDIA_TYPE:", metadata["MEDIA_TYPE"])


Parquet input: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_productivity_correction
STAC output: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\crop_productivity_correction
Collection id: crop_productivity_correction
Parquet stores: ['parquets/baseline/corrected_yield.parquet']
Temporal extent: ['2014-01-01T00:00:00', '2016-12-31T00:00:00']
Spatial extent: [103.41797806300008, 8.380953215000092, 106.82566220200007, 11.03326806100009]
MEDIA_TYPE: application/vnd.apache.parquet


## 2) Define STAC helper functions

Same building blocks as `12_crop_production.ipynb`, adapted for GeoParquet assets (table media type, bbox from geometry).


In [3]:
COLUMN_DESCRIPTIONS = {
    "area_map_name": "Administrative / freshwater-zone name (join key)",
    "crop_name": "Crop / season name used in RIBASIM mapping",
    "crop_name_fao": "FAO crop name used for salt tolerance / prices",
    "salinity": "Median salinity over crop area (dS/m or model units)",
    "yield": "Uncorrected production",
    "hectares": "Crop area (ha)",
    "year": "Calendar year",
    "a": "FAO salinity threshold (ECe)",
    "b": "FAO yield decline slope (% per dS/m)",
    "corrected_yield": "Salinity-corrected production",
    "corrected_yield_pp": "Corrected production valued with producer prices",
    "comment": "Correction notes",
    "FTE": "Full-time equivalent labour indicator",
    "Name": "Polygon name from province_fwzone",
    "geometry": "Polygon geometry (EPSG:4326)",
}


def parse_license(metadata: dict) -> str:
    license_ = metadata["LICENSE"]
    if "Creative Commons" in license_ and "4.0" in license_:
        return "CC-BY-4.0"
    return license_


def create_providers(metadata: dict) -> list[pystac.Provider]:
    return [
        pystac.Provider(
            name=metadata["PROVIDERS"]["name"],
            roles=[
                pystac.provider.ProviderRole.PRODUCER,
                pystac.provider.ProviderRole.LICENSOR,
            ],
            url=metadata["PROVIDERS"]["url"],
        ),
        pystac.Provider(
            name="Deltares",
            roles=[
                pystac.provider.ProviderRole.PROCESSOR,
                pystac.provider.ProviderRole.HOST,
            ],
            url="https://deltares.nl",
        ),
    ]


def create_extent(metadata: dict) -> pystac.Extent:
    start_datetime = datetime.datetime.strptime(
        metadata["TEMPORAL_EXTENT"][0].split("T")[0], "%Y-%m-%d"
    )
    end_datetime = None
    if len(metadata["TEMPORAL_EXTENT"]) > 1 and metadata["TEMPORAL_EXTENT"][1]:
        end_datetime = datetime.datetime.strptime(
            metadata["TEMPORAL_EXTENT"][1].split("T")[0], "%Y-%m-%d"
        )
    return pystac.Extent(
        pystac.SpatialExtent([metadata["SPATIAL_EXTENT"]]),
        pystac.TemporalExtent([[start_datetime, end_datetime]]),
    )


def bbox_geometry_from_extent(spatial_extent: list[float]) -> tuple[dict, list[float]]:
    west, south, east, north = spatial_extent
    bbox = [west, south, east, north]
    geometry = {
        "type": "Polygon",
        "coordinates": [
            [
                [west, south],
                [east, south],
                [east, north],
                [west, north],
                [west, south],
            ]
        ],
    }
    return geometry, bbox


def table_columns_from_geodataframe(gdf: gpd.GeoDataFrame) -> list[dict]:
    cols = []
    for name in gdf.columns:
        if name == "geometry":
            dtype = "geometry"
        else:
            dtype = str(gdf[name].dtype)
        cols.append(
            {
                "name": name,
                "type": dtype,
                "description": COLUMN_DESCRIPTIONS.get(name, ""),
            }
        )
    return cols


def create_collection(
    metadata: dict,
    collection_id: str,
    *,
    template_fp: Path,
) -> pystac.Collection:
    extent = create_extent(metadata)
    collection = get_template_collection(
        template_fp=str(template_fp),
        collection_id=collection_id,
        title=metadata["TITLE"],
        description=metadata["DESCRIPTION"],
        keywords=metadata["KEYWORDS"],
        license=parse_license(metadata),
        spatial_extent=[metadata["SPATIAL_EXTENT"]],
        temporal_extent=extent.temporal.intervals,
        providers=create_providers(metadata),
    )

    pystac.extensions.item_assets.ItemAssetsExtension.add_to(collection)
    collection.extra_fields["item_assets"] = {
        "data": {
            "type": metadata.get("MEDIA_TYPE", PARQUET_MEDIA_TYPE),
            "title": metadata["TITLE"],
            "roles": ["data"],
            "description": metadata["DESCRIPTION"],
        }
    }
    return collection


def apply_coclico_collection_props(collection: pystac.Collection, metadata: dict) -> None:
    schema_uri = CoclicoExtension.get_schema_uri()
    if collection.stac_extensions is None:
        collection.stac_extensions = [schema_uri]
    elif not any(
        uri.endswith("json-schema/schema.json") for uri in collection.stac_extensions
    ):
        collection.stac_extensions.append(schema_uri)

    coclico_ext = CollectionCoclicoExtension(collection)
    coclico_ext.units = metadata["UNITS"]
    coclico_ext.plot_series = PLOT_SERIES
    coclico_ext.plot_x_axis = PLOT_X_AXIS
    coclico_ext.plot_type = PLOT_TYPE
    coclico_ext.min_ = 0
    coclico_ext.linear_gradient = []


def parquet_storage_href(store: dict, href_prefix: str) -> str:
    return urljoin(href_prefix, store["filename"])


def process_parquet_store(
    store: dict,
    parquet_path: Path,
    metadata: dict,
    href_prefix: str,
) -> pystac.Item:
    """Build one STAC Item for a GeoParquet file (NUTS0-inspired, 12-style)."""
    gdf = gpd.read_parquet(parquet_path)
    if gdf.crs is not None and gdf.crs.to_epsg() != 4326:
        gdf_bounds = gdf.to_crs(epsg=4326)
    else:
        gdf_bounds = gdf

    minx, miny, maxx, maxy = map(float, gdf_bounds.total_bounds)
    bbox = [minx, miny, maxx, maxy]
    geometry = shapely.geometry.mapping(shapely.geometry.box(*bbox))

    if "year" in gdf.columns and gdf["year"].notna().any():
        year0 = int(gdf["year"].min())
        item_datetime = datetime.datetime(year0, 1, 1)
    else:
        item_datetime = datetime.datetime.strptime(
            metadata["TEMPORAL_EXTENT"][0].split("T")[0], "%Y-%m-%d"
        )

    item = pystac.Item(
        id=store["id"],
        geometry=geometry,
        bbox=bbox,
        datetime=item_datetime,
        properties={
            "title": store["title"],
            "description": store["description"],
            "table:row_count": int(len(gdf)),
            "table:columns": table_columns_from_geodataframe(gdf),
        },
    )
    item.common_metadata.created = datetime.datetime.now(datetime.timezone.utc)
    item.properties["deltares:item_key"] = store["id"]

    # Projection extension (lightweight)
    item.stac_extensions = item.stac_extensions or []
    proj_uri = "https://stac-extensions.github.io/projection/v1.1.0/schema.json"
    table_uri = "https://stac-extensions.github.io/table/v1.2.0/schema.json"
    for uri in (proj_uri, table_uri):
        if uri not in item.stac_extensions:
            item.stac_extensions.append(uri)
    item.properties["proj:epsg"] = 4326

    storage_href = parquet_storage_href(store, href_prefix)
    item.add_asset(
        "data",
        pystac.Asset(
            href=storage_href,
            media_type=PARQUET_MEDIA_TYPE,
            title=store["title"],
            description=store["description"],
            roles=["data"],
        ),
    )
    return item


def verify_local_stac_outputs(
    stac_dir: Path, collection_id: str, *, expected_items: int
) -> None:
    collection_json = stac_dir / collection_id / "collection.json"
    item_jsons = [
        p
        for p in (stac_dir / collection_id).rglob("*.json")
        if p.name != "collection.json"
    ]
    catalog_has_child = False
    catalog_path = stac_dir / "catalog.json"
    if catalog_path.is_file():
        with open(catalog_path, encoding="utf-8") as f:
            catalog_has_child = (
                f"./{collection_id}/collection.json" in json.dumps(json.load(f))
            )

    missing = []
    if not collection_json.is_file():
        missing.append(str(collection_json))
    if len(item_jsons) < expected_items:
        missing.append(
            f"{expected_items} item JSON files (found {len(item_jsons)})"
        )
    if not catalog_has_child:
        missing.append(f"catalog child link ./{collection_id}/collection.json")

    if missing:
        raise FileNotFoundError(
            "STAC outputs incomplete after save. Re-run steps 3–6, then step 7.\n"
            "Missing: " + "; ".join(missing)
        )

    print(f"Verified: {collection_json}")
    print(f"Verified: {len(item_jsons)} item JSON file(s)")
    print(f"Verified: catalog.json links to ./{collection_id}/collection.json")


## 3) Create the STAC collection


In [4]:
template_fp = stac_dir / "template" / "collection.json"
collection = create_collection(metadata, collection_id, template_fp=template_fp)
apply_coclico_collection_props(collection, metadata)
print(f"Collection ready: {collection.id}")
print("Temporal extent:", metadata["TEMPORAL_EXTENT"])


Collection ready: crop_productivity_correction
Temporal extent: ['2014-01-01T00:00:00', '2016-12-31T00:00:00']


## 4) Table metadata

Column descriptions are attached per item in `process_parquet_store` (`table:columns`, `table:row_count`). No Zarr datacube step.


In [5]:
preview = gpd.read_parquet(data_dir / PARQUET_STORES[0]["filename"])
display(pd.DataFrame(table_columns_from_geodataframe(preview)))
print("Preview rows:", len(preview))


,name,type,description
0,Name,object,Polygon name from province_fwzone
1,OBJECTID,int64,
2,zone,object,
3,crop_affec,float64,
4,geometry,geometry,Polygon geometry (EPSG:4326)
5,area_map_name,object,Administrative / freshwater-zone name (join key)
6,crop_name,object,Crop / season name used in RIBASIM mapping
7,crop_name_fao,object,FAO crop name used for salt tolerance / prices
8,salinity,float64,Median salinity over crop area (dS/m or model ...
9,yield,float64,Uncorrected production


Preview rows: 228


## 5) Build one test STAC item


In [6]:
test_store = PARQUET_STORES[0]
test_item = process_parquet_store(
    test_store,
    data_dir / test_store["filename"],
    metadata,
    href_prefix,
)
print("Test item id:", test_item.id)
print("Test item datetime:", test_item.datetime)
print("Data asset href:", test_item.assets["data"].href)
print("table:row_count:", test_item.properties.get("table:row_count"))
test_item


Test item id: baseline
Test item datetime: 2014-01-01 00:00:00
Data asset href: https://storage.googleapis.com/gca-data-public/gca/crop_productivity_correction/parquets/baseline/corrected_yield.parquet
table:row_count: 228


<Item id=baseline>

## 6) Build all STAC items


In [7]:
items = []
item_rows = []
item_errors = []

for store in PARQUET_STORES:
    print("now working on:", store["id"])
    parquet_path = data_dir / store["filename"]
    try:
        item = process_parquet_store(store, parquet_path, metadata, href_prefix)
        collection.add_item(item)
        items.append(item)
        item_rows.append(
            {
                "item_id": item.id,
                "rows": item.properties.get("table:row_count"),
                "href": item.assets["data"].href,
            }
        )
    except Exception as exc:
        item_errors.append({"store": store["id"], "error": str(exc)})
        print("  ERROR:", exc)

display(pd.DataFrame(item_rows))
if item_errors:
    display(pd.DataFrame(item_errors))


now working on: baseline


,item_id,rows,href
0,baseline,228,https://storage.googleapis.com/gca-data-public...


## 7) Save STAC catalog locally


In [8]:
stac_io = DefaultStacIO()
layout = pystac.layout.BestPracticesLayoutStrategy()

for store in PARQUET_STORES:
    (stac_dir / collection_id / store["id"]).mkdir(parents=True, exist_ok=True)

collection.update_extent_from_items()

catalog = pystac.Catalog.from_file(str(stac_dir / "catalog.json"))
if catalog.get_child(collection.id):
    catalog.remove_child(collection.id)
    print(f"Removed existing child: {collection.id}")

catalog.add_child(collection)
collection.normalize_hrefs(str(stac_dir / collection_id), strategy=layout)
catalog.save(
    catalog_type=pystac.CatalogType.SELF_CONTAINED,
    dest_href=str(stac_dir),
    stac_io=stac_io,
)

collection.validate_all()
catalog.validate_all()
verify_local_stac_outputs(stac_dir, collection_id, expected_items=len(PARQUET_STORES))
print(f"STAC saved to: {stac_dir}")


Verified: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\crop_productivity_correction\collection.json
Verified: 1 item JSON file(s)
Verified: catalog.json links to ./crop_productivity_correction/collection.json
STAC saved to: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current


## 8) Upload GeoParquet to Google Cloud (optional)

Publish each parquet to `gs://{bucket_name}/{bucket_proj}/{proj_name}/{filename}` matching STAC asset hrefs.


In [9]:
import gcsfs

load_google_credentials(google_token_fp=google_cred_path)
fs = gcsfs.GCSFileSystem(
    gcs_project, token=os.environ["GOOGLE_APPLICATION_CREDENTIALS"]
)

for store in PARQUET_STORES:
    local_path = data_dir / store["filename"]
    target = urljoin(bucket_name, bucket_proj, proj_name, store["filename"])
    print(f"Uploading {store['filename']} -> {target} ...")
    # Ensure parent prefix exists; put file at exact key
    parent = "/".join(target.split("/")[:-1])
    if fs.exists(target):
        print(f"  Removing existing remote file: {target}")
        fs.rm(target)
    fs.put(str(local_path), target)
    print(f"Done: {store['id']}")


Google Application Credentials load into environment.
Uploading parquets/baseline/corrected_yield.parquet -> gca-data-public/gca/crop_productivity_correction/parquets/baseline/corrected_yield.parquet ...


C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:230: FutureWarning: This function will be deprecated in the future, please use environment variables instead. When Google cloud is installed on your computer credentials can set using 'GOOGLE_DEFAULT' in the storage_kwargs argument
  warnings.warn(
C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\coclicodata\src\coclicodata\etl\cloud_utils.py:243: CredentialLeakageWarning: Keys loaded from shared network drive.
  warnings.warn(


Done: baseline


## 9) Upload STAC catalog to Google Cloud (optional)

Same pattern as `12_crop_production.ipynb`: collection folder + root `catalog.json`.


In [10]:
verify_local_stac_outputs(stac_dir, collection_id, expected_items=len(PARQUET_STORES))

load_google_credentials(google_token_fp=google_cred_path)

dir_to_google_cloud(
    dir_path=str(stac_dir / collection_id),
    gcs_project=gcs_project,
    bucket_name=bucket_name,
    bucket_proj=bucket_proj,
    dir_name=urljoin(stac_cloud_name, collection_id),
)

file_to_google_cloud(
    file_path=str(stac_dir / "catalog.json"),
    gcs_project=gcs_project,
    bucket_name=bucket_name,
    bucket_proj=bucket_proj,
    dir_name=stac_cloud_name,
    file_name="catalog.json",
)

print(
    f"Uploaded STAC collection: "
    f"gs://{bucket_name}/{bucket_proj}/{stac_cloud_name}/{collection_id}/"
)
print(
    f"Uploaded catalog: "
    f"gs://{bucket_name}/{bucket_proj}/{stac_cloud_name}/catalog.json"
)


Verified: C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data\current\crop_productivity_correction\collection.json
Verified: 1 item JSON file(s)
Verified: catalog.json links to ./crop_productivity_correction/collection.json
Google Application Credentials load into environment.
Writing to directory at gca-data-public/gca/gca-stac-7/crop_productivity_correction...
Done!
Uploaded STAC collection: gs://gca-data-public/gca/gca-stac-7/crop_productivity_correction/
Uploaded catalog: gs://gca-data-public/gca/gca-stac-7/catalog.json
